# 実習：5種類の花を分類し、代表的なCNNの転移学習を比較しよう
### CNN 第3週 ｜ PyTorch・Google Colab

**前半はResNet18で一つずつ確認し、後半は同じ処理を5モデルへ広げます。**

| 前半：仕組みを理解する | 後半：同じ条件で比較する |
|---|---|
| 花の画像・ラベルを見る | VGG16 / ResNet18 / Inception v3 |
| 学習済みResNet18を読み込む | DenseNet121 / EfficientNet-B0 |
| 5クラスのHeadに交換し、残りをFreeze | 同じTrain・Validation画像を使用 |
| Head学習前と学習後を比較 | Accuracy・時間・Parameter数を比較 |

**基本実習は「最後のLinear層だけを学習する」転移学習です。** CNN全体のFine-tuningではありません。

授業スライド28〜33ページの「同じ正解クラスで比較」「Train / Validation / Testを分ける」
という方針を、今回は**花の5クラス**に適用します。ImageNetの1,000クラス番号と花の番号は直接比較しません。

コードの暗記は不要です。各セルの直前にある説明と、実行後の形・数値・図を確認してください。

## 0. 実行の準備と読み方

Colabの **ランタイム → ランタイムのタイプを変更 → GPU** を選び、上から実行します。
CPUでも動作しますが、5モデルの比較にはGPUを推奨します。

- 花の画像とImageNet学習済み重みは、初回にインターネットから取得します。
- データ配布元はTensorFlowの公式サーバーですが、**TensorFlow / Kerasは使用しません**。
- VGG16の重みは特に大きいため、ダウンロード中は進捗表示を確認してください。
- 最初は `FAST_MODE = True`。Trainの一部を使い、処理の流れを確認します。
- **Testの結果を見て設定を直さない**こと。改善案はValidationで比べます。

必要なライブラリは通常のColabに含まれています。PyTorchとtorchvisionだけを別々に更新すると
組合せの不整合を起こす場合があるため、このノートブックでは無条件の `pip install -U` を行いません。

In [ ]:
import gc
import json
import random
import sys
import time
import tarfile
from pathlib import Path
from urllib.error import URLError

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageFont, ImageOps
from IPython.display import display
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torchvision
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, models
from torchvision.datasets.utils import download_and_extract_archive
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("torchvision:", torchvision.__version__)
print("Device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("CPUで実行します。5モデルの比較ではGPUの利用を推奨します。")

## 1. 比較条件を最初に決める

まず全画像を約 **70% / 15% / 15%** に分割します。
FAST MODEでは、**Train側から各クラス最大100枚（計500枚）だけ**を使います。
ValidationとTestは減らしません。したがってFAST MODEの「実際に学習する枚数」は全体の70%ではありません。

| 設定 | FAST MODE | 通常モード |
|---|---|---|
| Train | 各クラス最大100枚 | 70%側の全画像 |
| Validation / Test | 15%側の全画像 / 15%側の全画像 | 同じ |
| Epoch数 | 3 | 5 |
| 更新する部分 | 新しい最後のLinear層だけ | 同じ |

時間を短縮するときは `MODELS_TO_COMPARE` を `['resnet18', 'efficientnet_b0']` にします。
初期設定は5モデルすべてです。**設定を変えたら、前半を含めて上からやり直してください。**

In [ ]:
SEED = 42
FAST_MODE = True
MAX_TRAIN_PER_CLASS = 100
EPOCHS = 3 if FAST_MODE else 5
BATCH_SIZE = 16
LEARNING_RATE = 0.001
NUM_WORKERS = 0  # 初学者向けに、並列読込に関する環境差を避ける

MODELS_TO_COMPARE = [
    "resnet18", "vgg16", "inception_v3", "densenet121", "efficientnet_b0"
]

DATA_ROOT = Path("./data")
DATA_DIR = DATA_ROOT / "flower_photos"
OUTPUT_DIR = Path("./flowers_results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

set_seed()
print("FAST_MODE:", FAST_MODE)
print("Epochs:", EPOCHS, "Batch size:", BATCH_SIZE)
print("比較モデル:", MODELS_TO_COMPARE)

# PART A — ResNet18で転移学習の流れを理解する

## 2. 5種類の花を自動ダウンロードする

使用データは **flower_photos**。3,670枚の写真が、5個のクラス別フォルダに入っています。
Oxford Flowers 102とは別のデータセットです。

```text
flower_photos/
├── daisy/        デイジー
├── dandelion/    タンポポ
├── roses/        バラ
├── sunflowers/   ヒマワリ
└── tulips/       チューリップ
```

下のセルは、公式URLからアーカイブを取得して展開します。展開済みなら再利用します。
`LICENSE.txt` に写真の作者・利用条件が記載されています。出典は末尾の参考資料[1]です。

In [ ]:
DATA_URL = (
    "https://storage.googleapis.com/download.tensorflow.org/"
    "example_images/flower_photos.tgz"
)
EXPECTED_IMAGES = 3670

def prepare_flowers():
    DATA_ROOT.mkdir(parents=True, exist_ok=True)
    if len(list(DATA_DIR.glob("*/*.jpg"))) == EXPECTED_IMAGES:
        print("ダウンロード済みの画像を使います。")
        return

    last_error = None
    for attempt in range(1, 4):
        try:
            print(f"データを取得します（試行 {attempt}/3）。")
            download_and_extract_archive(
                DATA_URL,
                download_root=str(DATA_ROOT),
                extract_root=str(DATA_ROOT),
                filename="flower_photos.tgz"
            )
            count = len(list(DATA_DIR.glob("*/*.jpg")))
            if count != EXPECTED_IMAGES:
                raise RuntimeError(f"画像数が想定と異なります: {count}")
            return
        except (OSError, RuntimeError, EOFError, tarfile.TarError) as error:
            last_error = error
            # 中断したダウンロードを、次の試行で再利用しない
            archive = DATA_ROOT / "flower_photos.tgz"
            if archive.exists():
                archive.unlink()
            if attempt < 3:
                time.sleep(2)

    raise RuntimeError(
        "花データを取得できませんでした。通信状態を確認してこのセルを再実行してください。"
        "または公式アーカイブを展開し、./data/flower_photos/ に配置してください。"
    ) from last_error

prepare_flowers()
print("画像フォルダ:", DATA_DIR.resolve())

## 3. ImageFolderで「画像 X」と「正解 y」を読み込む

`ImageFolder` は、フォルダ名をクラス名として画像を読み込む仕組みです[2]。
この段階ではまだTensorへの変換を指定せず、元の写真を確認します。

**ラベル番号を暗記せず、`class_to_idx` で対応を確認してください。**
TensorFlow Datasets版のラベル順とは異なる場合があります。

In [ ]:
raw_dataset = datasets.ImageFolder(DATA_DIR)
class_names = raw_dataset.classes
NUM_CLASSES = len(class_names)
targets = np.array(raw_dataset.targets)

expected_classes = ["daisy", "dandelion", "roses", "sunflowers", "tulips"]
assert class_names == expected_classes, class_names
assert NUM_CLASSES == 5

print("全画像数:", len(raw_dataset))
print("クラスとラベル:", raw_dataset.class_to_idx)

data_summary = pd.DataFrame({
    "Label": range(NUM_CLASSES),
    "Class": class_names,
    "Images": np.bincount(targets, minlength=NUM_CLASSES)
})
display(data_summary)

## 4. 各クラスの写真を見る

1枚の画像が入力 **X**、花の種類を表す整数が正解 **y** です。
背景、花の向き、撮影距離が異なっていても、同じ花の種類を予測したいと考えます。

下のコードは**表示用**です。写真の一覧を作る処理を暗記する必要はありません。

In [ ]:
font_path = Path(matplotlib.get_data_path()) / "fonts/ttf/DejaVuSans.ttf"
font = ImageFont.truetype(str(font_path), 18)
sheet = Image.new("RGB", (230 * NUM_CLASSES, 260), "white")
draw = ImageDraw.Draw(sheet)

for label, name in enumerate(class_names):
    index = int(np.flatnonzero(targets == label)[0])
    image, _ = raw_dataset[index]
    thumbnail = ImageOps.fit(image, (220, 220))
    sheet.paste(thumbnail, (230 * label + 5, 5))
    draw.text((230 * label + 10, 232), f"{label}: {name}", fill="black", font=font)

display(sheet)

## 5. Train / Validation / Testを一度だけ分ける

- **Train**：Headの重みを学習する。
- **Validation**：学習前後を比べ、Epochやモデルを選ぶ。
- **Test**：選択が終わったあとに、最終モデルを評価する。

`stratify` で各花の割合をできるだけ保ちます。5モデルすべてで**同じ画像番号**を使います。
FAST MODEで使わなかったTrain画像をValidationやTestへ移すことはしません。

In [ ]:
all_indices = np.arange(len(raw_dataset))

train_indices_full, holdout_indices = train_test_split(
    all_indices,
    test_size=0.30,
    stratify=targets,
    random_state=SEED
)
val_indices, test_indices = train_test_split(
    holdout_indices,
    test_size=0.50,
    stratify=targets[holdout_indices],
    random_state=SEED
)

train_indices = train_indices_full.copy()
if FAST_MODE:
    rng = np.random.default_rng(SEED)
    selected = []
    for label in range(NUM_CLASSES):
        candidates = train_indices_full[targets[train_indices_full] == label]
        selected.extend(rng.choice(
            candidates,
            size=min(MAX_TRAIN_PER_CLASS, len(candidates)),
            replace=False
        ).tolist())
    train_indices = np.array(sorted(selected))

# 画像が別の役割に重複していないことを確認する
assert set(train_indices_full).isdisjoint(val_indices)
assert set(train_indices_full).isdisjoint(test_indices)
assert set(val_indices).isdisjoint(test_indices)
assert len(train_indices_full) + len(val_indices) + len(test_indices) == len(raw_dataset)

split_table = pd.DataFrame({
    "Class": class_names,
    "Train (70% pool)": np.bincount(targets[train_indices_full], minlength=NUM_CLASSES),
    "Train (used)": np.bincount(targets[train_indices], minlength=NUM_CLASSES),
    "Validation": np.bincount(targets[val_indices], minlength=NUM_CLASSES),
    "Test": np.bincount(targets[test_indices], minlength=NUM_CLASSES)
})
display(split_table)
print("実際の学習枚数:", len(train_indices))
print("Validation枚数:", len(val_indices), "/ Test枚数:", len(test_indices))

# 分割をCSVに記録する。後で同じ条件を確認できる。
manifest = pd.DataFrame({
    "path": [str(Path(path).relative_to(DATA_DIR)) for path, _ in raw_dataset.samples],
    "label": targets,
    "split": "train",
    "used_for_training": False
})
manifest.loc[val_indices, "split"] = "validation"
manifest.loc[test_indices, "split"] = "test"
manifest.loc[train_indices, "used_for_training"] = True
manifest.to_csv(OUTPUT_DIR / "data_split.csv", index=False)

## 6. ImageNet学習済みResNet18を読み込む

`weights=...` は**構造だけでなく、すでに学習された重みも読み込む**指定です[3]。
今回は将来 `DEFAULT` の内容が変わっても条件を確認できるよう、`IMAGENET1K_V1` を明示します。

この時点の最後の層 `fc` はImageNetの**1,000クラス用**です。まだ花5種類用ではありません。

In [ ]:
set_seed()
resnet_weights = models.ResNet18_Weights.IMAGENET1K_V1

load_start = time.perf_counter()
resnet_model = models.resnet18(weights=resnet_weights).to(device)
resnet_load_seconds = time.perf_counter() - load_start

print("読み込んだ重み:", resnet_weights)
print("変更前のfc:", resnet_model.fc)

## 7. 学習済み重みに合った前処理を使う

`weights.transforms()` には、その重みに対応したResize・CenterCrop・Tensor化・Normalizeが入っています[4]。
ResNet18では、最終的に `3 × 224 × 224` のTensorにします。

**モデル比較を単純にするため、今回はTrainにも同じ固定前処理を使い、ランダムな画像拡張はしません。**
これは「転移学習では画像拡張を使えない」という意味ではありません。

下の `make_loaders()` は、モデルごとに前処理だけを変えながら、同じ画像分割を再利用する補助関数です。

In [ ]:
def make_loaders(preprocess):
    dataset = datasets.ImageFolder(DATA_DIR, transform=preprocess)
    assert dataset.class_to_idx == raw_dataset.class_to_idx
    assert dataset.samples == raw_dataset.samples

    generator = torch.Generator().manual_seed(SEED)
    train_loader = DataLoader(
        Subset(dataset, train_indices.tolist()),
        batch_size=BATCH_SIZE,
        shuffle=True,
        generator=generator,
        num_workers=NUM_WORKERS,
        pin_memory=(device.type == "cuda")
    )
    val_loader = DataLoader(
        Subset(dataset, val_indices.tolist()),
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=(device.type == "cuda")
    )
    return train_loader, val_loader

resnet_preprocess = resnet_weights.transforms()
print(resnet_preprocess)

resnet_train_loader, resnet_val_loader = make_loaders(resnet_preprocess)
image, label = raw_dataset[int(train_indices[0])]
print("変換後の画像shape:", resnet_preprocess(image).shape)

## 8. 参考：元の1,000クラスの予測を一度見てみる

Train側の写真1枚を使い、まだ変更していないImageNetモデルの上位5候補を表示します。
ここで出る名前は、花5クラスのラベルとは違う名前の集合です。

**この出力番号と花の正解番号を比較してAccuracyを計算してはいけません。**
これは「モデルを呼び出せたか」の確認であり、花分類の学習前精度ではありません。

In [ ]:
resnet_model.eval()
preview_image, _ = raw_dataset[int(train_indices[0])]

with torch.no_grad():
    imagenet_logits = resnet_model(
        resnet_preprocess(preview_image).unsqueeze(0).to(device)
    )
    top_indices = imagenet_logits[0].topk(5).indices.cpu().tolist()

print("元の出力shape:", tuple(imagenet_logits.shape))
display(pd.DataFrame({
    "Rank": range(1, 6),
    "ImageNet class": [resnet_weights.meta["categories"][i] for i in top_indices]
}))
del imagenet_logits

## 9. Freezeして、最後のHeadを5クラス用に交換する

```text
画像 → ImageNet学習済み部分【固定】 → 新しいLinear層【学習】 → 花5クラス
```

1. 既存Parameterの `requires_grad` を `False` にする。
2. `fc` を新しい `nn.Linear(..., 5)` に交換する。
3. 新しい層のParameterは初期状態で学習可能なので、**新しいHeadだけ**が更新される[3]。

今回の **Headは「最後のLinear層」** という意味で統一します。
VGG16でも最後のLinear層だけを交換し、それ以前の全結合層も固定します。

In [ ]:
for parameter in resnet_model.parameters():
    parameter.requires_grad = False

in_features = resnet_model.fc.in_features
resnet_model.fc = nn.Linear(in_features, NUM_CLASSES)
resnet_model = resnet_model.to(device)
resnet_head = resnet_model.fc

print("変更後のfc:", resnet_head)
print("学習するParameter:")
for name, parameter in resnet_model.named_parameters():
    if parameter.requires_grad:
        print(" ", name, tuple(parameter.shape))

resnet_total_params = sum(p.numel() for p in resnet_model.parameters())
resnet_trainable_params = sum(p.numel() for p in resnet_model.parameters() if p.requires_grad)
print("全Parameter数:", f"{resnet_total_params:,}")
print("学習するParameter数:", f"{resnet_trainable_params:,}")

## 10. Freezeの注意：BatchNormの状態も変えない

`requires_grad=False` だけでは、BatchNormのrunning mean / varianceの更新までは止まりません。
そこで今回のHead学習中は次のようにします[5]。

```python
model.eval()   # 固定する部分のBatchNorm・Dropout等を評価動作にする
head.train()   # 新しいHeadだけを学習動作にする
```

**`eval()` は「勾配を止める」命令ではありません。** Headはこの状態でも `backward()` で学習できます。
一方、`torch.no_grad()` は勾配を記録しない設定です[6]。学習するHeadまで `no_grad()` で囲まないでください。

| 場面 | モード | 勾配 |
|---|---|---|
| 今回のHead学習 | 全体 `eval()` → Headのみ `train()` | Headに必要 |
| Validation / Test | 全体 `eval()` | `no_grad()` で不要にする |

既存部分のDropout等も固定した動作にするため、この実習は通常の「全層をtrainモードにする学習」と異なります。

## 11. 学習と評価に使う二つの関数

第1・2週と同じ考え方です。`train_head_one_epoch()` と `evaluate()` は**この教材で作った関数名**で、
PyTorchに最初から用意された命令ではありません。

学習は `zero_grad → forward → loss → backward → step` の順番です。
評価はWeightを変更せず、予測と正解を比べます。正解ラベルは0〜4の整数のまま使います。

`CrossEntropyLoss` に渡すのはSoftmax前のlogitsです。モデルの最後にSoftmaxを追加しません。

In [ ]:
def train_head_one_epoch(model, head, loader, optimizer, criterion):
    model.eval()  # 固定部分のBatchNormやDropoutを変化させない
    head.train()
    total_loss, total_correct, total = 0.0, 0, 0

    for images, labels in tqdm(loader, desc="Train", leave=False):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        count = labels.size(0)
        total_loss += loss.item() * count
        total_correct += (logits.argmax(dim=1) == labels).sum().item()
        total += count

    return total_loss / total, total_correct / total


def evaluate(model, loader, criterion):
    model.eval()
    total_loss, total_correct, total = 0.0, 0, 0
    actual, predicted = [], []

    with torch.no_grad():
        for images, labels in tqdm(loader, desc="Evaluate", leave=False):
            images, labels = images.to(device), labels.to(device)
            logits = model(images)
            loss = criterion(logits, labels)
            predictions = logits.argmax(dim=1)

            count = labels.size(0)
            total_loss += loss.item() * count
            total_correct += (predictions == labels).sum().item()
            total += count
            actual.extend(labels.cpu().tolist())
            predicted.extend(predictions.cpu().tolist())

    return {
        "loss": total_loss / total,
        "accuracy": total_correct / total,
        "actual": np.array(actual),
        "predicted": np.array(predicted)
    }

## 12. 新しいHeadを学習する前のAccuracyを測る

ここから比較するのは、**花5クラスにそろえたモデル**です。
Headはランダムな初期値なので、花の特徴とクラスの対応をまだ学んでいません。

5クラスの一様ランダム予測の期待値は20%ですが、この未学習Headの実測Accuracyが必ず20%になるわけではありません。
これは元のImageNetモデルの能力を測る値ではなく、**新しいHeadの出発点**です。

In [ ]:
criterion = nn.CrossEntropyLoss()
resnet_before = evaluate(resnet_model, resnet_val_loader, criterion)
print(f"Head学習前 Validation Accuracy: {resnet_before['accuracy'] * 100:.2f}%")
print(f"Head学習前 Validation Loss: {resnet_before['loss']:.4f}")

## 13. EpochごとにHeadを学習する

下の `fit_head()` は、同じ学習を複数モデルで繰り返すためのまとめ役です。
中核は次の順番だけです。

```text
for epoch ...
    TrainでHeadを更新する
    ValidationでAccuracyを確認する
    最も良かったHeadを保存する
```

**同じHeadの学習を続けます。EpochごとにHeadを作り直しません。**
Validation Accuracyが同じ場合は、Validation Lossが小さいEpochを選びます。
最後にそのEpochのHeadへ戻します。Testはまだ使いません。

保存するのは小さなHeadの重みだけです。固定部分の重みは元のImageNet重みから復元できます。

In [ ]:
def synchronize_device():
    if device.type == "cuda":
        torch.cuda.synchronize()


def fit_head(model, head, train_loader, val_loader):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(head.parameters(), lr=LEARNING_RATE)
    history = []
    best_accuracy, best_loss = -1.0, float("inf")
    best_state, best_epoch = None, 0

    synchronize_device()
    start = time.perf_counter()
    for epoch in range(1, EPOCHS + 1):
        train_loss, train_accuracy = train_head_one_epoch(
            model, head, train_loader, optimizer, criterion
        )
        validation = evaluate(model, val_loader, criterion)
        history.append({
            "Epoch": epoch,
            "Train loss": train_loss,
            "Train accuracy": train_accuracy,
            "Val loss": validation["loss"],
            "Val accuracy": validation["accuracy"]
        })

        is_better = (
            validation["accuracy"] > best_accuracy
            or (validation["accuracy"] == best_accuracy and validation["loss"] < best_loss)
        )
        if is_better:
            best_accuracy = validation["accuracy"]
            best_loss = validation["loss"]
            best_epoch = epoch
            best_state = {k: v.detach().cpu().clone() for k, v in head.state_dict().items()}

        print(
            f"Epoch {epoch}/{EPOCHS} | "
            f"Train loss {train_loss:.3f} / acc {train_accuracy * 100:.1f}% | "
            f"Val loss {validation['loss']:.3f} / acc {validation['accuracy'] * 100:.1f}%"
        )

    synchronize_device()
    elapsed_seconds = time.perf_counter() - start
    head.load_state_dict(best_state)
    model.eval()
    return pd.DataFrame(history), best_state, best_epoch, elapsed_seconds

In [ ]:
resnet_history, resnet_head_state, resnet_best_epoch, resnet_train_seconds = fit_head(
    resnet_model,
    resnet_head,
    resnet_train_loader,
    resnet_val_loader
)

resnet_after = evaluate(resnet_model, resnet_val_loader, criterion)
print("採用したEpoch:", resnet_best_epoch)
print(f"Head学習前 Val Accuracy: {resnet_before['accuracy'] * 100:.2f}%")
print(f"Head学習後 Val Accuracy: {resnet_after['accuracy'] * 100:.2f}%")
print(f"学習＋各EpochのValidation時間: {resnet_train_seconds:.1f} 秒")

## 14. 学習曲線と、同じ画像の予測の変化を見る

左から右へEpochが進みます。Trainは学習中の平均Accuracy、ValidationはそのEpoch終了時のAccuracyです。
数値が同じ条件の評価ではないため、差だけで過学習を断定しないでください。

下の表は**同じValidation画像**について、Head学習前と学習後の予測を比べます。
Test画像ではありません。

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(resnet_history["Epoch"], 100 * resnet_history["Train accuracy"], marker="o", label="Train (during epoch)")
plt.plot(resnet_history["Epoch"], 100 * resnet_history["Val accuracy"], marker="o", label="Validation")
plt.xlabel("Epoch")
plt.ylabel("Accuracy (%)")
plt.xticks(resnet_history["Epoch"])
plt.ylim(0, 100)
plt.title("ResNet18: head training")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

assert np.array_equal(resnet_before["actual"], resnet_after["actual"])
preview_count = min(10, len(val_indices))
resnet_prediction_table = pd.DataFrame({
    "Actual": [class_names[i] for i in resnet_after["actual"][:preview_count]],
    "Before head training": [class_names[i] for i in resnet_before["predicted"][:preview_count]],
    "After head training": [class_names[i] for i in resnet_after["predicted"][:preview_count]]
})
display(resnet_prediction_table)

## ここまでの確認

**新しく学習したのはどこでしょうか？** 最後の `fc` のWeightとBiasだけです。
元の畳み込み層だけでなく、BatchNormの保存統計も固定して使いました。

**何の前後を比較したでしょうか？** ImageNetの精度ではなく、同じ花5クラスのHead学習前後です。

ここまでが前半です。次は、この流れを別のCNNに置き換えます。

# PART B — 5つの代表的なCNNを比較する

## 15. モデルごとに「交換するHead」と「入力サイズ」が違う

| モデル | 交換する最後の層 | Head入力の特徴数 | 画像の最終サイズ |
|---|---|---:|---:|
| ResNet18 | `fc` | 512 | 224×224 |
| VGG16 | `classifier[6]` | 4096 | 224×224 |
| Inception v3 | `fc` | 2048 | 299×299 |
| DenseNet121 | `classifier` | 1024 | 224×224 |
| EfficientNet-B0 | `classifier[1]` | 1280 | 224×224 |

公式torchvision実装と指定した重みに対応する設定です[4,7–10]。
**VGG16の前段の全結合層も学習済み特徴の一部として固定**します。全classifierを学習する実験ではありません。

**Inception v3は注意が必要です。** 重みを通常どおり読み込んだ後に、今回は使わない補助分類器を無効化します。
主出力だけで5クラスHeadを学習します。重みが持つ入力変換は保持します[8]。

同じ画像分割・Epoch・学習率を使いますが、モデルの前処理・特徴次元・学習済み重みの作り方は異なります。
したがって、これは**実用的な転移学習設定の比較**であり、構造だけの完全な優劣比較ではありません。

In [ ]:
MODEL_SPECS = {
    "resnet18": (models.resnet18, models.ResNet18_Weights.IMAGENET1K_V1),
    "vgg16": (models.vgg16, models.VGG16_Weights.IMAGENET1K_V1),
    "inception_v3": (models.inception_v3, models.Inception_V3_Weights.IMAGENET1K_V1),
    "densenet121": (models.densenet121, models.DenseNet121_Weights.IMAGENET1K_V1),
    "efficientnet_b0": (models.efficientnet_b0, models.EfficientNet_B0_Weights.IMAGENET1K_V1)
}
assert "resnet18" in MODELS_TO_COMPARE, "前半のResNet18は比較に残してください。"
assert set(MODELS_TO_COMPARE).issubset(MODEL_SPECS)


def build_transfer_model(model_name):
    builder, weights = MODEL_SPECS[model_name]
    model = builder(weights=weights)

    if model_name == "inception_v3":
        # 事前学習重みを読み込んだ後に、補助出力だけを除く
        model.aux_logits = False
        model.AuxLogits = None

    for parameter in model.parameters():
        parameter.requires_grad = False

    if model_name in ("resnet18", "inception_v3"):
        model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)
        head = model.fc
    elif model_name == "vgg16":
        model.classifier[6] = nn.Linear(model.classifier[6].in_features, NUM_CLASSES)
        head = model.classifier[6]
    elif model_name == "densenet121":
        model.classifier = nn.Linear(model.classifier.in_features, NUM_CLASSES)
        head = model.classifier
    else:  # efficientnet_b0
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, NUM_CLASSES)
        head = model.classifier[1]

    model = model.to(device)
    model.eval()
    return model, head, weights

## 16. 比較結果を保存する準備

ResNet18は前半の結果をそのまま使います。後半で同じ学習を繰り返しません。

記録する項目は、学習前後のValidation Accuracy、最良Epoch、Parameter数、処理時間です。
`Train+Val (s)` は学習と各EpochのValidationを含む時間です。ダウンロードは含みません。
`Load (s)` はモデルの生成・重み読込・GPU転送の時間で、初回ダウンロードも含む場合があります。

**モデルは一つずつGPUへ載せます。** 終了後はモデル本体を解放し、小さなHeadの重みと結果だけを保存します。

In [ ]:
results = {}
histories = {}
head_states = {}
validation_predictions = {}


def record_result(name, model, head, weights, before, after, history, state, best_epoch, train_seconds, load_seconds):
    preprocess = weights.transforms()
    results[name] = {
        "Model": name,
        "Weights": f"{type(weights).__name__}.{weights.name}",
        "Input crop": preprocess.crop_size[0],
        "Features": head.in_features,
        "Before Val (%)": before["accuracy"] * 100,
        "After Val (%)": after["accuracy"] * 100,
        "Gain (pp)": (after["accuracy"] - before["accuracy"]) * 100,
        "Best epoch": best_epoch,
        "Val loss": after["loss"],
        "Total params": sum(p.numel() for p in model.parameters()),
        "Trainable params": sum(p.numel() for p in model.parameters() if p.requires_grad),
        "Train+Val (s)": train_seconds,
        "Load (s)": load_seconds
    }
    histories[name] = history
    head_states[name] = state
    validation_predictions[name] = after

    torch.save(state, OUTPUT_DIR / f"{name}_head.pth")
    history.to_csv(OUTPUT_DIR / f"{name}_history.csv", index=False)
    pd.DataFrame(results.values()).to_csv(OUTPUT_DIR / "model_comparison.csv", index=False)


record_result(
    "resnet18", resnet_model, resnet_head, resnet_weights,
    resnet_before, resnet_after, resnet_history, resnet_head_state,
    resnet_best_epoch, resnet_train_seconds, resnet_load_seconds
)

# 他モデルを載せる前に、前半のモデル本体を解放する
resnet_model.cpu()
del resnet_model, resnet_head, resnet_train_loader, resnet_val_loader
gc.collect()
if device.type == "cuda":
    torch.cuda.empty_cache()
print("ResNet18の結果を保存し、モデル本体を解放しました。")

## 17. 同じ学習・評価を別のモデルにも適用する

下の関数は、前半の処理をそのまままとめたものです。

```text
学習済みモデルを読む → Freeze・Head交換 → 学習前Val → Head学習 → 学習後Val → 保存
```

ダウンロードに失敗した場合は、そのモデルのセルで停止します。
**学習済み重みを取得できないとき、勝手にランダム重みへ切り替えることはしません。**
正常に終わったモデルの結果は保存されているので、エラー解消後は失敗したモデルのセルから再実行できます。

In [ ]:
def run_comparison_model(name):
    if name not in MODELS_TO_COMPARE:
        print(name, "は今回の比較対象から外しています。")
        return

    print("=" * 60)
    print("比較モデル:", name)
    set_seed()
    start = time.perf_counter()
    model, head, weights = build_transfer_model(name)
    load_seconds = time.perf_counter() - start

    try:
        train_loader, val_loader = make_loaders(weights.transforms())
        criterion = nn.CrossEntropyLoss()
        before = evaluate(model, val_loader, criterion)
        print(f"Head学習前 Val Accuracy: {before['accuracy'] * 100:.2f}%")

        history, state, best_epoch, train_seconds = fit_head(
            model, head, train_loader, val_loader
        )
        after = evaluate(model, val_loader, criterion)
        record_result(
            name, model, head, weights, before, after, history, state,
            best_epoch, train_seconds, load_seconds
        )
        print(f"Head学習後 Val Accuracy: {after['accuracy'] * 100:.2f}%")
        print("Best epoch:", best_epoch)
    finally:
        # 途中でエラーになってもGPUメモリを解放する
        model.cpu()
        del model, head
        gc.collect()
        if device.type == "cuda":
            torch.cuda.empty_cache()

### 17-A. VGG16

3×3の畳み込みを重ねたモデルです。最後の `classifier[6]` を5クラスに交換します。
ダウンロードと計算の負担が比較的大きい点も観察します。前段の全結合層は固定します。

In [ ]:
run_comparison_model("vgg16")

### 17-B. Inception v3

複数の処理経路を使うモデルです。入力が299×299となる点に注目します。
補助出力はこの実習では使いません。ResNet18と同じ学習関数で、主Headだけを学習します。

In [ ]:
run_comparison_model("inception_v3")

### 17-C. DenseNet121

以前の層の特徴を連結して再利用するモデルです。最後の `classifier` を交換します。

In [ ]:
run_comparison_model("densenet121")

### 17-D. EfficientNet-B0

深さ・幅・解像度のバランスを考えた系列の小型モデルです。最後の `classifier[1]` を交換します。
モデルが小さいことと、今回のデータでAccuracyが高いことは別々に確認してください。

In [ ]:
run_comparison_model("efficientnet_b0")

## 18. 5モデルの結果を一つの表で比べる

`Before Val (%)` と `After Val (%)` は、**同じValidation画像**でのHead学習前後のAccuracyです。
`Gain (pp)` はパーセントポイント差です。例えば20%→80%なら +60ポイントです。

低かった学習前精度からの伸びが大きいモデルが、学習後に最も良いモデルとは限りません。
主に **After Val (%)** を見て、時間・Parameter数と併せて判断します。

In [ ]:
missing = [name for name in MODELS_TO_COMPARE if name not in results]
if missing:
    raise RuntimeError(f"未実行または未完了のモデルがあります: {missing}")

comparison = pd.DataFrame([results[name] for name in MODELS_TO_COMPARE])
columns = [
    "Model", "Input crop", "Before Val (%)", "After Val (%)", "Gain (pp)",
    "Best epoch", "Total params", "Trainable params", "Train+Val (s)"
]
display(comparison[columns].round(2))
comparison.to_csv(OUTPUT_DIR / "model_comparison.csv", index=False)

## 19. Accuracyと処理時間を図で確認する

Accuracyの図と時間の図は、単位が違うため別々に表示します。
入力解像度、前処理、CPU / GPU、画像読込にも時間がかかります。
**この時間は純粋な推論速度のベンチマークではありません。**

In [ ]:
x = np.arange(len(comparison))
plt.figure(figsize=(10, 5))
plt.bar(x - 0.18, comparison["Before Val (%)"], width=0.36, label="Before head training")
plt.bar(x + 0.18, comparison["After Val (%)"], width=0.36, label="After head training")
plt.xticks(x, comparison["Model"], rotation=20, ha="right")
plt.ylabel("Validation Accuracy (%)")
plt.ylim(0, 100)
plt.title("Flowers 5: before and after transfer learning")
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "accuracy_comparison.png", dpi=160)
plt.show()

plt.figure(figsize=(10, 4))
plt.bar(comparison["Model"], comparison["Train+Val (s)"])
plt.xticks(rotation=20, ha="right")
plt.ylabel("Seconds")
plt.title("Training + per-epoch validation time (download excluded)")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "time_comparison.png", dpi=160)
plt.show()

## 20. Validation Accuracyの学習曲線を見る

同じEpoch数でも、学習の進み方はモデルによって異なります。
途中の最高値だけでなく、改善が続いているか、変動が大きいかを確認します。
3 Epochの結果だけで、モデル一般の優劣を結論づけないでください。

In [ ]:
plt.figure(figsize=(9, 5))
for name in MODELS_TO_COMPARE:
    history = histories[name]
    plt.plot(history["Epoch"], 100 * history["Val accuracy"], marker="o", label=name)
plt.xlabel("Epoch")
plt.ylabel("Validation Accuracy (%)")
plt.xticks(range(1, EPOCHS + 1))
plt.ylim(0, 100)
plt.title("Validation learning curves")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "validation_curves.png", dpi=160)
plt.show()

# PART C — 選んだモデルをTestで最後に確認する

## 21. Validationだけでモデルを選ぶ

今回は、学習後Validation Accuracyが最大のモデルを選びます。
同点のときはValidation Lossが小さい方を選びます。

選んだモデルの**最良EpochのHead**を復元して、まだ予測に使っていないTestを評価します。
手順を簡単にするため、この教材ではTrainとValidationを合体して再学習する段階は設けません。
**同じHeadを復元するため、比較時のTrain / Validation分割と重みの版を維持します。**

Testを見てから「別モデルを試そう」「Epochを変更しよう」と繰り返さないでください。

In [ ]:
ranked = comparison.sort_values(
    ["After Val (%)", "Val loss", "Model"],
    ascending=[False, True, True]
)
selected_name = ranked.iloc[0]["Model"]
print("Validationで選んだモデル:", selected_name)

selected_model, selected_head, selected_weights = build_transfer_model(selected_name)
selected_head.load_state_dict(head_states[selected_name])
selected_model.eval()

final_dataset = datasets.ImageFolder(DATA_DIR, transform=selected_weights.transforms())
assert final_dataset.samples == raw_dataset.samples
final_test_loader = DataLoader(
    Subset(final_dataset, test_indices.tolist()),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(device.type == "cuda")
)

# Testの推論はこの1回。後の表・図でも同じ結果を再利用する。
test_result = evaluate(selected_model, final_test_loader, nn.CrossEntropyLoss())
print(f"Final Test Accuracy: {test_result['accuracy'] * 100:.2f}%")
print(f"Final Test Loss: {test_result['loss']:.4f}")

## 22. Confusion Matrixで間違い方を見る

行が正解、列が予測です。対角線以外が誤分類です。
Accuracy一つだけでは分からない、クラスごとの得意・不得意を確認します。

ここでは**保存したTest予測**を使います。モデルを再学習・再選択する処理ではありません。

In [ ]:
cm = confusion_matrix(
    test_result["actual"], test_result["predicted"], labels=list(range(NUM_CLASSES))
)
display_cm = ConfusionMatrixDisplay(cm, display_labels=class_names)
fig, ax = plt.subplots(figsize=(7, 6))
display_cm.plot(ax=ax, colorbar=False, xticks_rotation=30)
ax.set_title(f"Test confusion matrix: {selected_name}")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "confusion_matrix.png", dpi=160)
plt.show()

per_class_recall = np.divide(
    np.diag(cm), cm.sum(axis=1),
    out=np.zeros(NUM_CLASSES, dtype=float), where=cm.sum(axis=1) != 0
)
display(pd.DataFrame({
    "Class": class_names,
    "Test images": cm.sum(axis=1),
    "Recall (%)": per_class_recall * 100
}).round(2))

## 23. 間違えた写真を見る

下の写真は、予測したTest画像の元写真です。実際のモデル入力にはResizeとCenterCropが適用されています。
花びらの形、色、背景、切り取り範囲などから、間違えた理由を考えてください。

**写真だけから本当の原因を断定せず、「考えられる理由」として説明します。**

In [ ]:
wrong = np.flatnonzero(test_result["actual"] != test_result["predicted"])
print("誤分類数:", len(wrong), "/", len(test_indices))

if len(wrong) == 0:
    print("このTestでは誤分類はありませんでした。")
else:
    chosen = np.random.default_rng(SEED).choice(wrong, size=min(5, len(wrong)), replace=False)
    wrong_sheet = Image.new("RGB", (250 * len(chosen), 280), "white")
    draw = ImageDraw.Draw(wrong_sheet)
    for column, position in enumerate(chosen):
        dataset_index = int(test_indices[position])
        image, actual_label = raw_dataset[dataset_index]
        predicted_label = int(test_result["predicted"][position])
        wrong_sheet.paste(ImageOps.fit(image, (240, 220)), (250 * column + 5, 5))
        draw.text((250 * column + 6, 230), f"Actual: {class_names[actual_label]}", fill="black", font=font)
        draw.text((250 * column + 6, 253), f"Pred: {class_names[predicted_label]}", fill="black", font=font)
    display(wrong_sheet)
    wrong_sheet.save(OUTPUT_DIR / "wrong_predictions.png")

## 24. 結果を保存する

比較表・学習曲線・Headの重み・データ分割を `flowers_results` に保存します。
Headの重みだけでは推論できないため、復元時は**同じモデル・同じImageNet重み・同じラベル順・同じ前処理**が必要です。

Colabの一時保存領域は、セッション終了で消える場合があります。提出前に必要なファイルを保存してください。

In [ ]:
metadata = {
    "dataset_url": DATA_URL,
    "classes": class_names,
    "class_to_idx": raw_dataset.class_to_idx,
    "seed": SEED,
    "fast_mode": FAST_MODE,
    "train_images_used": len(train_indices),
    "validation_images": len(val_indices),
    "test_images": len(test_indices),
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "optimizer": "Adam",
    "augmentation": False,
    "trained_part": "last Linear only",
    "models_compared": MODELS_TO_COMPARE,
    "selected_model": selected_name,
    "selected_weights": results[selected_name]["Weights"],
    "test_accuracy": float(test_result["accuracy"]),
    "torch_version": str(torch.__version__),
    "torchvision_version": str(torchvision.__version__),
    "device": str(device)
}
(OUTPUT_DIR / "experiment_settings.json").write_text(
    json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8"
)
pd.DataFrame({
    "path": [str(Path(raw_dataset.samples[int(i)][0]).relative_to(DATA_DIR)) for i in test_indices],
    "actual_label": test_result["actual"],
    "predicted_label": test_result["predicted"]
}).to_csv(OUTPUT_DIR / "final_test_predictions.csv", index=False)
print("保存先:", OUTPUT_DIR.resolve())
print("保存したファイル:")
for path in sorted(OUTPUT_DIR.iterdir()):
    print(" ", path.name)

### 任意：結果をZIPで保存する

下のセルは `DOWNLOAD_RESULTS = True` にしたときだけ実行します。ノートブック本体はColabのファイルメニューから別に保存してください。

In [ ]:
DOWNLOAD_RESULTS = False
if DOWNLOAD_RESULTS:
    import shutil
    zip_path = shutil.make_archive("flowers_results", "zip", root_dir=OUTPUT_DIR)
    try:
        from google.colab import files
        files.download(zip_path)
    except ImportError:
        print("ZIPファイル:", Path(zip_path).resolve())

# まとめ・確認問題

## 今日行ったこと

**前半**：ResNet18を読む → 新しい5クラスHeadへ交換 → 固定部分をFreeze → Headを学習 → 前後を比較。

**後半**：同じ画像分割で5モデルを比較 → Validationで選択 → 選んだモデルだけをTestで確認。

## 確認問題

1. 入力 **X** と正解 **y** は何ですか。各クラスの名前とラベル番号を書いてください。
2. Train / Validation / Testは何のために分けましたか。FAST MODEで実際に使った枚数も書いてください。
3. ResNet18のHeadを1,000出力から5出力へ変えた理由は何ですか。
4. `requires_grad=False` と `model.eval()` は何が違いますか。なぜ今回、両方を使いましたか。
5. ResNet18のHead学習前と学習後のValidation Accuracyを記録してください。
6. 5モデルの比較表から、最も良いValidation Accuracyと、最も短い処理時間のモデルを挙げてください。両者は同じでしたか。
7. Inception v3の前処理では、他モデルと何が違いましたか。
8. 最後に選んだモデルのTest Accuracyと、混同されやすかった花の組合せを記録してください。
9. 「Head学習前より後の方が良い」という結果だけで、事前学習の効果を完全に証明できますか。
10. この比較から「一番新しいCNNは、どんなデータでも一番良い」と結論づけられますか。理由も書いてください。

**提出例**：実行済みipynb、`model_comparison.csv`、比較グラフ、Confusion Matrix、確認問題への回答。
AIは利用して構いません。結果の数値は実行結果から取り、自分の言葉で説明してください。

# 発展実験は基本実習のあとで

このノートブックの基本実習は、**最後のLinear層だけの学習**で完結します。
Fine-tuningやランダム初期値との比較を、基本実習の結果と混ぜないようにしてください。

- **一部Fine-tuning**：ResNet18の `layer4` と `fc` を学習可能にし、小さい学習率で追加学習する。
  この場合はOptimizerの対象も変える必要があり、今回のHead専用関数をそのまま使ってはいけません。
- **事前学習の効果**：同じ構造・同じ学習範囲・同じ予算で、ランダム初期値から学ぶ場合と比較する。
  ランダムなBackboneを固定した比較は「ランダム特徴の利用」であり、CNN全体を最初から学習する実験ではありません。
- **画像数の効果**：FAST MODEから通常モードへ変え、Validationで比較する。

追加の比較もValidationで行います。今回のTestを見て何度も改良した場合、そのTestは独立した最終評価とは呼べなくなります。

## うまく動かないとき

| 症状 | 確認すること |
|---|---|
| データの取得に失敗 | 通信を確認し、データ取得セルを再実行。公式アーカイブを展開して配置する方法も使える |
| モデル重みの取得に失敗 | 通信を確認して、そのモデルのセルを再実行。`weights=None` へ勝手に変えない |
| CUDA out of memory | 他のモデルが残っていないか確認。ランタイムを再起動し、`BATCH_SIZE=8` で最初から実行 |
| `torchvision::...` 等のimportエラー | 新しいColabランタイムで再実行。PyTorchとtorchvisionの不整合を疑う |
| Headの形が合わない | `NUM_CLASSES=5` と、交換前Headの `in_features` を確認 |
| 正確度が低い／モデル順が予想と違う | FAST MODE、少ないEpoch、画像数、固定特徴と対象画像の相性を確認 |

設定を変えた実験では、結果フォルダも分けると取り違えを防げます。
固定Seedでも、ライブラリやGPUが異なると数値が完全一致しない場合があります。

## 参考・実験条件の位置づけ

**授業の方針**：配布資料『CNN_Week3_CNN_TransferLearning_Theory.pdf』28〜33ページ。
対象クラスをそろえること、学習前後と事前学習効果を区別すること、Validationで比較してTestを最後に使うことを引き継いでいます。
5クラス花データ、FAST MODE、Epoch・学習率、最終Linear層のみという具体的な設定は、この実習のために選んだ条件です。

**データ・APIの一次資料**

1. [TensorFlow公式：flower_photosの配布・画像数・ライセンス](https://www.tensorflow.org/tutorials/load_data/images)
2. [torchvision ImageFolder](https://docs.pytorch.org/vision/stable/generated/torchvision.datasets.ImageFolder.html)
3. [PyTorch公式：Transfer Learning Tutorial](https://docs.pytorch.org/tutorials/beginner/transfer_learning_tutorial.html)
4. [torchvision ResNet18・学習済み重みと前処理](https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.resnet18.html)
5. [PyTorch Module：eval / requires_grad](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html)
6. [PyTorch no_grad](https://docs.pytorch.org/docs/stable/generated/torch.no_grad.html)
7. [torchvision VGG16](https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.vgg16.html)
8. [torchvision Inception v3（実装）](https://docs.pytorch.org/vision/stable/_modules/torchvision/models/inception.html)
9. [torchvision DenseNet121](https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.densenet121.html)
10. [torchvision EfficientNet-B0](https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.efficientnet_b0.html)

画像は実行時に公式サーバーから取得します。再配布や公開の際は、画像アーカイブ内の `LICENSE.txt` を確認してください。